# FABRIC Slice Visualizer — Demo

This notebook demonstrates both `fabvis` visualizers using **mock data** so you can test the UI without a live FABRIC connection.

Two visualizers are shown:
1. **SliceVisualizer** — Cytoscape graph topology (nodes, components, networks as an interactive graph)
2. **GeoVisualizer** — Geographic map view (sites on a world map with animated network links)

The mock topology includes:
- Multiple nodes across sites (RENC, STAR, UTAH)
- NICs, GPUs, NVMe storage
- L2 Bridge and FABNetv4 networks
- A facility port
- Two slices

## Mock FABlib Objects

These mock classes mimic the FABlib API so the visualizers work without a real FABRIC connection.

In [ ]:
class MockInterface:
    def __init__(self, name, ip=None, vlan=None, mac=None, bandwidth=None, node=None, network=None, site=None):
        self._name = name; self._ip = ip; self._vlan = vlan; self._mac = mac
        self._bw = bandwidth; self._node = node; self._network = network; self._site = site
    def get_name(self): return self._name
    def get_ip_addr(self): return self._ip
    def get_vlan(self): return self._vlan
    def get_mac(self): return self._mac
    def get_bandwidth(self): return self._bw
    def get_site(self): return self._site
    def get_physical_os_interface_name(self): return "eth1"
    def get_reservation_state(self): return "Active"
    def get_subnet(self): return "10.0.0.0/24" if self._ip else None
    def get_node(self): return self._node
    def get_network(self): return self._network

class MockComponent:
    def __init__(self, name, model, interfaces=None, node=None):
        self._name = name; self._model = model
        self._interfaces = interfaces or []; self._node = node
    def get_name(self): return self._name
    def get_model(self): return self._model
    def get_type(self): return self._model.split("_")[0]
    def get_details(self): return f"{self._model} component"
    def get_pci_addr(self): return "0000:41:00.0"
    def get_numa_node(self): return "0"
    def get_device_name(self): return None
    def get_interfaces(self): return self._interfaces
    def get_node(self): return self._node

class MockNode:
    def __init__(self, name, site="RENC", cores=4, ram=16, disk=100,
                 image="default_rocky_9", components=None, interfaces=None, state="Active"):
        self._name = name; self._site = site; self._cores = cores; self._ram = ram
        self._disk = disk; self._image = image; self._state = state
        self._components = components or []; self._interfaces = interfaces or []
    def get_name(self): return self._name
    def get_site(self): return self._site
    def get_host(self): return f"{self._site.lower()}-w1.fabric-testbed.net"
    def get_cores(self): return self._cores
    def get_ram(self): return self._ram
    def get_disk(self): return self._disk
    def get_image(self): return self._image
    def get_management_ip(self): return f"2001:db8::{hash(self._name) % 256}"
    def get_reservation_state(self): return self._state
    def get_username(self): return "rocky"
    def get_components(self): return self._components
    def get_interfaces(self): return self._interfaces

class MockFacilityPort:
    def __init__(self, name, site, interfaces=None):
        self._name = name; self._site = site; self._ifaces = interfaces or []
    def get_name(self): return self._name
    def get_site(self): return self._site
    def get_model(self): return "Facility_Port"
    def get_host(self): return None
    def get_cores(self): return None
    def get_ram(self): return None
    def get_disk(self): return None
    def get_image(self): return None
    def get_management_ip(self): return None
    def get_reservation_state(self): return "Active"
    def get_username(self): return None
    def get_components(self): return []
    def get_interfaces(self): return self._ifaces

class MockNetworkService:
    def __init__(self, name, net_type="L2Bridge", interfaces=None, subnet=None, gateway=None, site=None):
        self._name = name; self._type = net_type
        self._interfaces = interfaces or []; self._subnet = subnet; self._gw = gateway; self._site = site
    def get_name(self): return self._name
    def get_type(self): return self._type
    def get_layer(self): return "L2" if "L2" in self._type else "L3"
    def get_site(self): return self._site or "RENC"
    def get_subnet(self): return self._subnet
    def get_gateway(self): return self._gw
    def get_interfaces(self): return self._interfaces

class MockSlice:
    def __init__(self, name, slice_id=None, state="StableOK",
                 nodes=None, networks=None, facilities=None):
        self._name = name; self._id = slice_id or f"id-{name}"
        self._state = state; self._nodes = nodes or []
        self._networks = networks or []; self._facilities = facilities or []
    def get_name(self): return self._name
    def get_slice_id(self): return self._id
    def get_state(self): return self._state
    def get_lease_end(self): return "2026-03-15 00:00:00 +0000"
    def get_nodes(self): return self._nodes
    def get_network_services(self): return self._networks
    def get_facilities(self): return self._facilities
    def get_interfaces(self): return []
    def get_components(self): return []

class MockFablib:
    """Mock FablibManager that returns pre-built slices."""
    def __init__(self, slices):
        self._slices = {s.get_name(): s for s in slices}
    def get_slices(self):
        return list(self._slices.values())
    def get_slice(self, name):
        if name not in self._slices:
            raise Exception(f"Slice '{name}' not found")
        return self._slices[name]

print("Mock classes defined.")

## Build Test Topology

**Slice 1: "web_application"** — 3-tier web app across two sites
- `frontend` @ RENC (4c/8GB) with NIC_Basic → L2 bridge
- `backend` @ RENC (8c/32GB) with NIC_ConnectX_6 + GPU_A40 → L2 bridge + FABNetv4
- `database` @ STAR (8c/64GB) with NIC_Basic + NVME_P4510 → FABNetv4

**Slice 2: "ml_training"** — GPU compute across 3 sites
- `trainer` @ UTAH (16c/64GB) with NIC_ConnectX_6 + GPU_A40 → FABNetv4
- `data_server` @ RENC (4c/16GB) with NIC_Basic → FABNetv4
- `gpu_worker` @ UCSD (8c/32GB) with NIC_Basic + GPU_A40 → FABNetv4
- Facility port at STAR

In [ ]:
# ── Slice 1: web_application ──

# Interfaces (site is set for geo visualizer to map them correctly)
fe_nic_iface   = MockInterface("frontend-nic1-p0", ip="10.0.0.1", vlan="100", mac="aa:bb:cc:00:00:01", site="RENC")
be_nic1_iface  = MockInterface("backend-nic1-p0",  ip="10.0.0.2", vlan="100", mac="aa:bb:cc:00:00:02", bandwidth=100, site="RENC")
be_nic2_iface  = MockInterface("backend-nic2-p0",  ip="10.1.0.1", mac="aa:bb:cc:00:00:03", site="RENC")
db_nic_iface   = MockInterface("database-nic1-p0", ip="10.1.0.2", mac="aa:bb:cc:00:00:04", site="STAR")

# Components
fe_nic  = MockComponent("nic1", "NIC_Basic", [fe_nic_iface])
be_nic1 = MockComponent("nic1", "NIC_ConnectX_6", [be_nic1_iface])
be_nic2 = MockComponent("nic2", "NIC_Basic", [be_nic2_iface])
be_gpu  = MockComponent("gpu1", "GPU_A40", [])
db_nic  = MockComponent("nic1", "NIC_Basic", [db_nic_iface])
db_nvme = MockComponent("nvme1", "NVME_P4510", [])

# Nodes
frontend = MockNode("frontend", "RENC", cores=4, ram=8, disk=50,
                    components=[fe_nic], interfaces=[fe_nic_iface])
backend  = MockNode("backend", "RENC", cores=8, ram=32, disk=100,
                    components=[be_nic1, be_nic2, be_gpu], interfaces=[be_nic1_iface, be_nic2_iface])
database = MockNode("database", "STAR", cores=8, ram=64, disk=500,
                    components=[db_nic, db_nvme], interfaces=[db_nic_iface])

# Wire up node references on interfaces
for iface in [fe_nic_iface]: iface._node = frontend
for iface in [be_nic1_iface, be_nic2_iface]: iface._node = backend
for iface in [db_nic_iface]: iface._node = database

# Networks
app_bridge = MockNetworkService("app_bridge", "L2Bridge",
                                [fe_nic_iface, be_nic1_iface], subnet="10.0.0.0/24", site="RENC")
mgmt_net   = MockNetworkService("mgmt_fabnet", "FABNetv4",
                                [be_nic2_iface, db_nic_iface], subnet="10.1.0.0/24", gateway="10.1.0.254")

slice_web = MockSlice("web_application", state="StableOK",
                      nodes=[frontend, backend, database],
                      networks=[app_bridge, mgmt_net])

# ── Slice 2: ml_training (3 sites) ──

tr_nic_iface = MockInterface("trainer-nic1-p0", ip="10.2.0.1", mac="aa:bb:cc:00:01:01", bandwidth=100, site="UTAH")
ds_nic_iface = MockInterface("data_server-nic1-p0", ip="10.2.0.2", mac="aa:bb:cc:00:01:02", site="RENC")
gw_nic_iface = MockInterface("gpu_worker-nic1-p0", ip="10.2.0.3", mac="aa:bb:cc:00:01:03", site="UCSD")
fp_iface     = MockInterface("cloudlab-port-p0", vlan="300", site="STAR")

tr_nic = MockComponent("nic1", "NIC_ConnectX_6", [tr_nic_iface])
tr_gpu = MockComponent("gpu1", "GPU_A40", [])
ds_nic = MockComponent("nic1", "NIC_Basic", [ds_nic_iface])
gw_nic = MockComponent("nic1", "NIC_Basic", [gw_nic_iface])
gw_gpu = MockComponent("gpu1", "GPU_A40", [])

trainer     = MockNode("trainer", "UTAH", cores=16, ram=64, disk=200,
                       components=[tr_nic, tr_gpu], interfaces=[tr_nic_iface])
data_server = MockNode("data_server", "RENC", cores=4, ram=16, disk=100,
                       components=[ds_nic], interfaces=[ds_nic_iface])
gpu_worker  = MockNode("gpu_worker", "UCSD", cores=8, ram=32, disk=200,
                       components=[gw_nic, gw_gpu], interfaces=[gw_nic_iface])

# Wire up node references
tr_nic_iface._node = trainer
ds_nic_iface._node = data_server
gw_nic_iface._node = gpu_worker

cloudlab_fp = MockFacilityPort("cloudlab_port", "STAR", [fp_iface])

compute_net = MockNetworkService("compute_fabnet", "FABNetv4",
                                 [tr_nic_iface, ds_nic_iface, gw_nic_iface],
                                 subnet="10.2.0.0/24", gateway="10.2.0.254")

slice_ml = MockSlice("ml_training", state="StableOK",
                     nodes=[trainer, data_server, gpu_worker],
                     networks=[compute_net],
                     facilities=[cloudlab_fp])

# Mock FablibManager
fablib = MockFablib([slice_web, slice_ml])

print(f"Built {len(fablib.get_slices())} slices:")
for s in fablib.get_slices():
    print(f"  {s.get_name()}: {len(s.get_nodes())} nodes, {len(s.get_network_services())} networks")

---
## 1. Cytoscape Topology Viewer — `SliceVisualizer`

Interactive graph-based view. Try these interactions:
- Select one or both slices, click **Load**
- **Click any node** (box) to see cores, RAM, site, components
- **Click a component** (small circle) to see model, PCI address
- **Click a network** (diamond) to see type, subnet, connected interfaces
- **Click an edge** (line) to see IP, VLAN, MAC, bandwidth
- Change **Layout** (dagre, breadthfirst, etc.)

In [ ]:
from fabrictestbed_extensions.fabvis import SliceVisualizer

vis = SliceVisualizer(fablib)
vis.show()

### Programmatic loading

Load both slices directly and use a hierarchical layout:

In [ ]:
vis2 = SliceVisualizer(fablib)
vis2.load_slices(["web_application", "ml_training"])
vis2.set_layout("dagre")
vis2.show()

---
## 2. Geographic Map Viewer — `GeoVisualizer`

Shows slices on an interactive world map. Try:
- Select slices and click **Load** to see nodes placed at their FABRIC sites
- **Click a blue site marker** to see what's at that site
- **Click a colored node marker** to see node details
- **Click an animated path** (ant path) to see the network connection
- Toggle **Show all sites** to hide/show background site markers
- Zoom into the US to see sites more clearly

In [ ]:
from fabrictestbed_extensions.fabvis import GeoVisualizer

geo = GeoVisualizer(fablib)
geo.show()

### Programmatic loading

Load both slices to see the multi-site network paths:

In [ ]:
geo2 = GeoVisualizer(fablib)
geo2.load_slices(["web_application", "ml_training"])
geo2.show()

### Single slice on the map

In [ ]:
geo3 = GeoVisualizer(fablib)
geo3.load_slices(["ml_training"])
geo3.show()